In [1]:
import torch

In [2]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(device)

mps


In [3]:
# torch.randn: returns a tensor filled with random numbers from a normal distribution with mean 0 and variance 1 (also called the standard normal distribution)

x = torch.randn(4, 4)
x = x.to(device=device)

y = torch.randn(4, 4, device=device)

model = torch.nn.Linear(10, 2)
model.to(device=device)

# RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!
# z = x + torch.randn(4, 4)

Linear(in_features=10, out_features=2, bias=True)

Interesting so you cannot do tensor operations when the tensor is on a different device (ex. CPU versus MPS) Fascinating!

In [4]:
# np.linspace 
# Return evenly spaced numbers over a specified interval. 

import numpy as np

arr = np.linspace(-2, 2, 512) # this makes arr a float 64 but mps doesn't support that, same with GPU they don't support 64
# t = torch.from_numpy(arr).to(device)
t = torch.from_numpy(arr).float().to(device)
# TypeError: Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.

In [5]:
# TLDR: GPU work is asynchronous

import time

big_tensor = torch.randn(100, 100)

start = time.perf_counter()
result = big_tensor @ big_tensor
torch.mps.synchronize()

delta = time.perf_counter() - start
delta

0.001991832861676812

In [8]:
import time
import torch

def get_device():
    return torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

def bench_matmul(n, device, reps=10):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)

    # warm-up (compiles Metal kernels / primes caches; NOT timed)
    for _ in range(3):
        _ = a @ b
    if device.type == "mps":
        torch.mps.synchronize()

    start = time.perf_counter()
    total = 0.0
    for _ in range(reps):
        c = a @ b
        total += c[0, 0].item()   # forces a real readback so the GPU can't skip the work
    if device.type == "mps":
        torch.mps.synchronize()
    elapsed = time.perf_counter() - start
    return elapsed / reps


if __name__ == "__main__":
    mps = get_device()
    cpu = torch.device("cpu")

    # one throwaway call per device to absorb process-level cold-start jitter
    # (thread pool spin-up, first-call OS/scheduler effects, etc.)
    bench_matmul(64, cpu)
    bench_matmul(64, mps)

    print(f"{'size':>6} | {'cpu (ms)':>10} | {'mps (ms)':>10} | speedup")
    print("-" * 46)
    for n in [64, 128, 256, 512, 1024, 2048, 4096]:
        t_cpu = bench_matmul(n, cpu) * 1000
        t_mps = bench_matmul(n, mps) * 1000
        print(f"{n:>6} | {t_cpu:>10.3f} | {t_mps:>10.3f} | {t_cpu / t_mps:>6.2f}x")

  size |   cpu (ms) |   mps (ms) | speedup
----------------------------------------------
    64 |      0.033 |      0.776 |   0.04x
   128 |      0.047 |      0.355 |   0.13x
   256 |      0.271 |      0.372 |   0.73x
   512 |      1.274 |      0.709 |   1.80x
  1024 |      9.274 |      1.605 |   5.78x
  2048 |     57.455 |      5.484 |  10.48x
  4096 |    418.743 |     41.188 |  10.17x


In [16]:
!python julia_mps_render.py

Rendering on: mps
  frame 0/90  zoom=1.0x
  frame 10/90  zoom=2.4x
  frame 20/90  zoom=5.6x
  frame 30/90  zoom=13.3x
  frame 40/90  zoom=31.4x
  frame 50/90  zoom=74.4x
  frame 60/90  zoom=176.0x
  frame 70/90  zoom=416.7x
  frame 80/90  zoom=986.6x
Rendered 90 frames in 13.62s
Saved julia_zoom.gif
